# 🕵️ Dia 06 — Registros DNS como evidência: o grafo de pegada de um domínio

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 6* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-06.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

16h30. Dois cooperados receberam um e-mail "da Serra Azul" com o remetente **verdadeiro**. Marta: *"Invadiram o nosso e-mail?"*

Rafa coletou o DNS dos três domínios. O DMARC da cooperativa é `p=none`. E os dois domínios do golpe, com IPs diferentes, têm no TXT o mesmo *token*: `ik-77c1e9a04b`.

Neste notebook você vai:

1. **Decodificar cada registro** (SOA com data, RNAME como e-mail, SPF, DMARC, MX nulo).
2. **Aplicar o fluxograma do DMARC** ao e-mail falso "da Serra Azul".
3. **Achar valores compartilhados** entre domínios, separando vínculo forte de infraestrutura comum.
4. **Gerar o grafo em DOT e Mermaid** e os achados com confiança.
5. **Jogar o quiz "o que este registro diz?"** (8 perguntas).

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-06/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['serraazul.example-raw.txt'] = '== A\nserraazul.example.      3600    IN      A       198.51.100.20\n== AAAA\n== NS\nserraazul.example.      86400   IN      NS      ns1.provedor-mg.example.\nserraazul.example.      86400   IN      NS      ns2.provedor-mg.example.\n== SOA\nserraazul.example.      3600    IN      SOA     ns1.provedor-mg.example. ti.serraazul.example. 2026081401 7200 3600 1209600 3600\n== MX\nserraazul.example.      3600    IN      MX      10 mx1.correio-coop.example.\nserraazul.example.      3600    IN      MX      20 mx2.correio-coop.example.\n== TXT\nserraazul.example.      3600    IN      TXT     "v=spf1 ip4:198.51.100.25 include:_spf.correio-coop.example ~all"\n== CAA\nserraazul.example.      3600    IN      CAA     0 issue "ac-exemplo.example"\n_dmarc.serraazul.example. 3600  IN      TXT     "v=DMARC1; p=none; rua=mailto:dmarc@serraazul.example"\n2026-10-05T19:30:12Z\n'
EMBUTIDO['serraazul-pix.example-raw.txt'] = '== A\nserraazul-pix.example.  300     IN      A       203.0.113.77\n== AAAA\n== NS\nserraazul-pix.example.  86400   IN      NS      ns1.dnsbarato.example.\nserraazul-pix.example.  86400   IN      NS      ns2.dnsbarato.example.\n== SOA\nserraazul-pix.example.  3600    IN      SOA     ns1.dnsbarato.example. hostmaster.dnsbarato.example. 2026100401 10800 3600 604800 300\n== MX\n== TXT\nserraazul-pix.example.  300     IN      TXT     "v=spf1 -all"\nserraazul-pix.example.  300     IN      TXT     "painel-verificacao=ik-77c1e9a04b"\n== CAA\n2026-10-05T19:31:40Z\n'
EMBUTIDO['serraazul-seguranca.example-raw.txt'] = '== A\nserraazul-seguranca.example. 300 IN     A       203.0.113.78\n== AAAA\n== NS\nserraazul-seguranca.example. 86400 IN   NS      ns1.dnsbarato.example.\nserraazul-seguranca.example. 86400 IN   NS      ns2.dnsbarato.example.\n== SOA\nserraazul-seguranca.example. 3600 IN    SOA     ns1.dnsbarato.example. hostmaster.dnsbarato.example. 2026100402 10800 3600 604800 300\n== MX\n== TXT\nserraazul-seguranca.example. 300 IN     TXT     "painel-verificacao=ik-77c1e9a04b"\n== CAA\n2026-10-05T19:33:05Z\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import hashlib, re
from collections import defaultdict

TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
for nome, txt in TEXTOS.items():
    print(f"{hashlib.sha256(txt.encode()).hexdigest()}  {nome}")       # custódia, sempre

def ler_coleta(texto):
    """Lê a saída do laço 'for t in A AAAA ...' do Passo 1. Devolve {tipo: [registros]} e a hora."""
    regs, hora = defaultdict(list), None
    for linha in texto.splitlines():
        if re.fullmatch(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ", linha.strip()):
            hora = linha.strip()                                   # a linha do date -u no fim
        elif linha and not linha.startswith("=="):
            nome, ttl, _in, tipo, dado = linha.split(None, 4)      # 5 colunas; o dado pode ter espaços
            if nome.startswith("_dmarc."):
                tipo = "DMARC"                                     # separamos o DMARC do TXT comum
            regs[tipo].append(dado.strip())
    return regs, hora

COLETAS = {nome.replace("-raw.txt", ""): ler_coleta(txt) for nome, txt in TEXTOS.items()}
for dom, (regs, hora) in COLETAS.items():
    print(f"\n== {dom}  (coletado {hora})")
    for tipo, valores in regs.items():
        for v in valores:
            print(f"   {tipo:6} {v}")

## 1. Decodificando SOA, SPF, DMARC e MX

In [ ]:
from datetime import date

def decodificar_soa(soa):
    """'ns1. hostmaster.dnsbarato.example. 2026100401 ...' → primário, e-mail, data do serial."""
    primario, rname, serial, *_ = soa.split()
    usuario, _, dominio = rname.rstrip(".").partition(".")      # o 1º ponto do RNAME é o @
    email = f"{usuario}@{dominio}"
    s = str(serial)
    try:
        data = date(int(s[:4]), int(s[4:6]), int(s[6:8]))        # formato AAAAMMDDnn (convenção, não regra)
        leitura = f"{data.isoformat()}, alteração nº {int(s[8:])} (inferência: é só uma convenção)"
    except ValueError:
        leitura = "não parece data"
    return primario, email, leitura

def postura_email(regs):
    """Resume MX, SPF e DMARC de um domínio."""
    mx = regs.get("MX", [])
    spf = next((t.strip('"') for t in regs.get("TXT", []) if "v=spf1" in t), None)
    dmarc = next((t.strip('"') for t in regs.get("DMARC", [])), None)
    p = re.search(r"\bp=(\w+)", dmarc).group(1) if dmarc else None
    return {
        "MX": mx or ["(nenhum MX: sem e-mail de entrada declarado)"],
        "SPF": spf or "(sem SPF)",
        "SPF final": (spf.split()[-1] if spf else "—"),
        "DMARC": dmarc or "(sem DMARC)",
        "política": p or "nenhuma",
    }

for dom, (regs, hora) in COLETAS.items():
    print(f"\n== {dom}")
    for soa in regs.get("SOA", []):
        prim, email, leitura = decodificar_soa(soa)
        print(f"   SOA: primário {prim} · admin {email} · serial {leitura}")
    for k, v in postura_email(regs).items():
        print(f"   {k}: {v}")

## 2. O e-mail falso "da Serra Azul", pelo fluxograma do DMARC

O fluxograma do original virou função. Rode com o domínio verdadeiro e veja o que um receptor que respeita DMARC faria com um e-mail falsificado.

In [ ]:
#@title Simular um e-mail falsificado { display-mode: "form" }
DOMINIO_FROM = "serraazul.example"  #@param ["serraazul.example", "serraazul-pix.example"]
PASSA_SPF_OU_DKIM_ALINHADO = False  #@param {type:"boolean"}

def destino_dmarc(regs, alinhado):
    """O fluxograma do Dia 6, passo a passo."""
    dmarc = next((t for t in regs.get("DMARC", [])), None)
    if not dmarc:
        return "Sem DMARC: nenhuma instrução do domínio; a filtragem do próprio receptor decide."
    if alinhado:
        return "DMARC aprovado: entregue normalmente."
    p = re.search(r"\bp=(\w+)", dmarc).group(1)
    return {"none": "p=none: só monitorar. O receptor NÃO é instruído a agir → o e-mail falso tende a chegar.",
            "quarantine": "p=quarantine: tratar como suspeito (ex.: pasta de spam).",
            "reject": "p=reject: rejeitar a mensagem."}[p]

print(f"From: atendimento@{DOMINIO_FROM}, enviado por terceiro")
print("→", destino_dmarc(COLETAS[DOMINIO_FROM][0], PASSA_SPF_OU_DKIM_ALINHADO))

> **Para a Marta, em cinco linhas:** *"O domínio de vocês publica uma política de e-mail que só pede monitoramento (`p=none`). Isso significa que os provedores de e-mail dos cooperados não são instruídos a recusar mensagens falsas com o remetente da Serra Azul. Qualquer pessoa consegue enviar um e-mail assim sem invadir nada. Isso não prova que o servidor de vocês está íntegro, mas explica o que aconteceu sem precisar de invasão. Recomendamos ler os relatórios DMARC por algumas semanas e subir para `quarantine` e depois `reject`."*

## 3. O que os domínios têm em comum?

Montamos um índice reverso: para cada **valor** (IP, NS, TXT...), quais domínios o usam. Depois separamos o que é **vínculo forte** (valor exclusivo do operador) do que é **infraestrutura comum** (compartilhada com milhares de clientes).

In [ ]:
indice = defaultdict(set)                  # (tipo, valor) → domínios que o usam
for dom, (regs, _) in COLETAS.items():
    for tipo, valores in regs.items():
        for v in valores:
            if tipo == "SOA":
                v = v.split()[1]           # do SOA, só o RNAME interessa para comparar
            indice[(tipo, v)].add(dom)

# Regra simples (e explícita) para a força do vínculo. Você pode discordar e mudar.
def forca(tipo, valor):
    if tipo == "TXT" and "=" in valor and "spf1" not in valor:
        return "FORTE (token exclusivo de verificação)"
    if tipo in ("NS", "SOA"):
        return "fraco (provedor de DNS atende muitos clientes)"
    if tipo == "TXT" and "spf1 -all" in valor:
        return "fraco (configuração genérica, comum em domínios sem e-mail)"
    return "médio (avaliar)"

VINCULOS = []
for (tipo, valor), doms in sorted(indice.items()):
    if len(doms) > 1:
        VINCULOS.append((tipo, valor, sorted(doms), forca(tipo, valor)))
        print(f"{tipo:4} {valor:45} {sorted(doms)}\n      → {forca(tipo, valor)}")

## 4. 🎮 Quiz: o que este registro diz?

Responda **S** (verdadeiro) ou **N** (falso). `MODO_JOGO = False` mostra só as respostas.

In [ ]:
QUIZ = [
    ("O SOA 'hostmaster.dnsbarato.example' indica o e-mail hostmaster@dnsbarato.example.", True,
     "O primeiro ponto do RNAME faz o papel do @."),
    ("O serial 2026100401 PROVA que a zona foi criada em 4/out/2026.", False,
     "Serial em formato de data é convenção, não regra. É inferência forte, não prova."),
    ("DMARC p=none instrui os receptores a rejeitar e-mails falsificados.", False,
     "p=none só pede monitoramento."),
    ("Um e-mail falso 'de' serraazul.example explica-se sem invasão do servidor da cooperativa.", True,
     "Sem política que mande rejeitar, falsificar o remetente não exige invadir nada."),
    ("Dois domínios usarem ns1.dnsbarato.example é um vínculo forte entre eles.", False,
     "Provedor de DNS barato atende milhares de clientes. É infraestrutura comum."),
    ("O mesmo token TXT de verificação em dois domínios sugere a mesma conta/operador.", True,
     "Tokens costumam ser únicos por conta. É vínculo forte (ainda assim, corrobore)."),
    ("'v=spf1 -all' sem MX significa que o domínio não deveria enviar nem receber e-mail.", True,
     "SPF que não autoriza ninguém e nenhum MX: domínio sem e-mail."),
    ("Para ver todos os registros, o melhor é 'dig ANY'.", False,
     "RFC 8482: a maioria dos servidores responde ANY com o mínimo. Consulte tipo por tipo."),
]
MODO_JOGO = True

acertos = 0
for i, (frase, verdade, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {frase}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Verdadeiro? (S/N): ").strip().upper()
        ok = (r == "S") == verdade
        acertos += ok
        print("   ✅ Acertou!" if ok else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {'verdadeiro' if verdade else 'falso'} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 6 else "— meta: 6 de 8.")

## 5. 📝 O grafo (DOT e Mermaid) e os achados

O grafo junta os três domínios. O rótulo traz hora UTC e resolvedor (requisito do Checkpoint). A aresta do *token* é desenhada em destaque.

In [ ]:
#@title Proveniência do grafo { display-mode: "form" }
RESOLVEDOR = "1.1.1.1"  #@param {type:"string"}

from pathlib import Path

horas = sorted(h for _, h in COLETAS.values())
rotulo = f"Pegada DNS Operação Isca, coletada {horas[0]} a {horas[-1]} via {RESOLVEDOR}"

arestas = []                                                  # (origem, destino, rótulo, destaque?)
for dom, (regs, _) in COLETAS.items():
    for ip in regs.get("A", []):
        arestas.append((dom, ip, "A", False))
    for ns in regs.get("NS", []):
        arestas.append((dom, ns, "NS", False))
    for mx in regs.get("MX", []):
        arestas.append((dom, mx.split()[1], f"MX {mx.split()[0]}", False))
    pe = postura_email(regs)
    if pe["SPF"] != "(sem SPF)":
        arestas.append((dom, f"SPF: {pe['SPF'].replace('v=spf1 ', '')}", "TXT", False))
    arestas.append((dom, f"DMARC: p={pe['política']}" if pe["política"] != "nenhuma" else "sem DMARC", "_dmarc TXT", False))
    caa = regs.get("CAA", [])
    arestas.append((dom, f"AC: {caa[0].split()[-1].strip(chr(34))}" if caa else "sem CAA", "CAA", False))
for tipo, valor, doms, f in VINCULOS:
    if f.startswith("FORTE"):
        token = valor.strip('"')
        for d in doms:
            arestas.append((d, f"token {token}", "mesmo token TXT", True))

# --- DOT (Graphviz) ---
dot = ["digraph footprint {", "  rankdir=LR;",
       '  node [shape=box, style="rounded,filled", fillcolor="#eef3fb", fontname="Helvetica"];',
       '  edge [fontname="Helvetica", fontsize=10];']
for d in COLETAS:
    cor = "#ffe8a8" if d == "serraazul.example" else "#f8c9c9"
    dot.append(f'  "{d}" [fillcolor="{cor}"];')
for a, b, r, destaque in arestas:
    extra = ', color="#d62728", penwidth=3' if destaque else ""
    dot.append(f'  "{a}" -> "{b}" [label="{r}"{extra}];')
dot += [f'  label="{rotulo}";', "  labelloc=b;", "}"]
DOT = "\n".join(dot)

# --- Mermaid ---
ids = {}
def nid(x):
    return ids.setdefault(x, f"N{len(ids)}")
mer = ["---", f"title: {rotulo}", "---", "graph LR"]
for i, (a, b, r, destaque) in enumerate(arestas):
    mer.append(f'    {nid(a)}["{a}"] -->|"{r}"| {nid(b)}["{b}"]')
destaques = [str(i) for i, x in enumerate(arestas) if x[3]]
if destaques:
    mer.append(f"    linkStyle {','.join(destaques)} stroke:#d62728,stroke-width:3px")
MERMAID = "\n".join(mer)

achados = [
    "DMARC p=none em serraazul.example (_dmarc TXT) → receptores não são instruídos a rejeitar e-mail falsificado com o remetente da cooperativa → alta, afirmado no registro",
    "SPF ~all em serraazul.example (TXT) → falha leve: e-mail de servidor não listado é só marcado → alta, afirmado no registro",
    "serraazul-pix.example e serraazul-seguranca.example compartilham o TXT painel-verificacao=ik-77c1e9a04b (TXT) → provavelmente verificados pela mesma conta de painel → média-alta; corroborar com certificado (Dia 8)",
    "Os dois domínios do golpe usam ns1/ns2.dnsbarato.example (NS) → mesmo provedor de DNS → baixa como vínculo: infraestrutura comum",
    "Serial do SOA de serraazul-pix.example = 2026100401 (SOA) → zona alterada em 4/out/2026, véspera do disparo → média, inferência pela convenção de data",
    "Domínios do golpe sem MX e com SPF -all (MX/TXT) → não foram configurados para receber e-mail; o e-mail falso usou o remetente da cooperativa, não estes domínios → média",
]
Path("notas").mkdir(exist_ok=True)
Path("notas/day06-footprint-isca.dot").write_text(DOT + "\n", encoding="utf-8")
Path("notas/day06-footprint-isca.md").write_text("```mermaid\n" + MERMAID + "\n```\n", encoding="utf-8")
Path("notas/day06-findings-isca.md").write_text("# Achados — pegada DNS (Operação Isca)\n\n" +
                                                "\n".join(f"- {a}" for a in achados) + "\n", encoding="utf-8")
print(MERMAID)
print("\n".join(["", "Achados:"] + achados))
print("\n💾 notas/day06-footprint-isca.dot · .md · day06-findings-isca.md")
print("   Para PNG: dot -Tpng notas/day06-footprint-isca.dot -o day06-footprint-isca.png")
try:
    from google.colab import files
    for f in ("notas/day06-footprint-isca.dot", "notas/day06-footprint-isca.md", "notas/day06-findings-isca.md"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta notas/)")

### (Opcional) Renderizar o DOT no Colab

Linha com `!` roda no terminal do Colab; fora dele, ignore.

In [ ]:
# Opcional: só funciona no Colab (ou em Linux com apt). O teste do repo pula linhas com "!".
!apt-get -qq install -y graphviz > /dev/null
!dot -Tpng notas/day06-footprint-isca.dot -o notas/day06-footprint-isca.png
try:
    from IPython.display import Image, display     # mostra a imagem, se existir
    display(Image("notas/day06-footprint-isca.png"))
except Exception as erro:
    print("Sem Graphviz aqui; use o Mermaid acima ou https://mermaid.live —", type(erro).__name__)

## ✅ Antes de fechar o Dia 6

- [ ] Explica à Marta o `p=none` em até cinco linhas
- [ ] Separa vínculo forte (token) de infraestrutura comum (NS)
- [ ] Datas dos seriais do golpe anotadas na linha do tempo do caso
- [ ] Quiz com pelo menos 6/8
- [ ] Fez também a prática original com `wikipedia.org` e `example.com`

**Amanhã (Dia 7):** o link do SMS era `lnk.example/sa7`. Do encurtador até o formulário, quantos saltos? E por que o `curl` viu "manutenção" e o celular viu um formulário de CPF?

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>